In [1]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from models import get_model, get_embeddings
from pdf_chunk import text_spliter

import re

e:\Rohanta_AI_workbook\adavance_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = get_model()
embedding_model = get_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3298.89it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# CELL 1 — Build sentence-level documents with window metadata
def build_sentence_window_docs(documents, window_size=3):
    """
    Split each document into sentences.
    Each sentence stores a window of surrounding sentences in metadata.
    """
    sentence_docs = []

    for doc in documents:
        # Split on sentence boundaries
        sentences = re.split(r'(?<=[.!?])\s+', doc.page_content.strip())
        sentences = [s.strip() for s in sentences if s.strip()]

        for i, sentence in enumerate(sentences):
            # Compute window boundaries
            start = max(0, i - window_size)
            end   = min(len(sentences), i + window_size + 1)
            window_text = " ".join(sentences[start:end])

            sentence_docs.append(Document(
                page_content=sentence,        # ← what gets EMBEDDED
                metadata={
                    **doc.metadata,
                    "window": window_text,    # ← what gets sent to LLM
                    "sentence_index": i,
                    "window_size": window_size,
                }
            ))

    return sentence_docs

In [ ]:
# CELL 2 — Index into Chroma
sentence_docs = build_sentence_window_docs(text_spliter, window_size=3)

print(f"Original chunks : {len(text_spliter)}")
print(f"Sentence docs   : {len(sentence_docs)}")
print(f"\nExample sentence : {sentence_docs[4].page_content}")
print(f"Example window   : {sentence_docs[4].metadata['window']}")



Original chunks : 49
Sentence docs   : 125

Example sentence : He is currently open to ML Engineer, AI Engineer,
Example window   : Rohanta Bhamare is an AI Engineer with 5+ years of experience building and deploying production-grade AI systems across deep learning, NLP, Large Language Models (LLMs), Generative AI, and computer vision. Previously an Assistant Professor in Computer Science, he brings both research-level depth and hands-on engineering expertise. He is skilled in MLOps, cloud infrastructure automation using AWS, Terraform, and Kubernetes, and CI/CD pipelines. He is currently open to ML Engineer, AI Engineer,


In [5]:
# Index — only the sentence goes into the embedding
vectorstore = Chroma.from_documents(
    sentence_docs,
    embedding_model,
    collection_name="sentence_window"
)

In [14]:
# CELL 3 — Retriever with window replacement
def sentence_window_retriever(query, k=7):
    """
    1. Search by sentence embedding (precise)
    2. Swap page_content with window (rich context)
    """
    # Step 1 — retrieve matched sentences
    matched = vectorstore.similarity_search(query, k=k)

    # Step 2 — replace content with window
    window_docs = []
    for doc in matched:
        window_text = doc.metadata.get("window", doc.page_content)
        window_docs.append(Document(
            page_content=window_text,   # ← LLM sees this
            metadata=doc.metadata
        ))

    return window_docs




In [15]:
docs = sentence_window_retriever("current university of rohanta")

In [16]:
for i in docs:
    print(i)

page_content='LangGraph enables graph-based AI workflows where nodes represent reasoning steps and edges represent execution transitions. This allows conditional branching, iterative reasoning loops, tool usage, stateful memory, and dynamic decision making. In agentic systems, the LLM can decide when to retrieve additional information, when to call external tools, when to validate outputs, and when to terminate reasoning. Rohanta actively experiments with' metadata={'window': 'LangGraph enables graph-based AI workflows where nodes represent reasoning steps and edges represent execution transitions. This allows conditional branching, iterative reasoning loops, tool usage, stateful memory, and dynamic decision making. In agentic systems, the LLM can decide when to retrieve additional information, when to call external tools, when to validate outputs, and when to terminate reasoning. Rohanta actively experiments with', 'source': 'my_data.txt', 'sentence_index': 4, 'window_size': 3}
page_c

In [17]:
# CELL 4 — Full LCEL chain

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.
If the answer is not in the context, say "I don't know".

Context: {context}
Question: {question}
""")

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

chain = (
    {
        "context": RunnableLambda(
            lambda q: format_docs(sentence_window_retriever(q, k=4))
        ),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("what is Rohanta's current university?"))

I don't know.
